In [1]:
import cv2
import numpy as np
import json
import os
import csv
import gdspy
from matplotlib.path import Path
import numpy as np
import pandas as pd
from collections import defaultdict
import math

In [2]:
# calculate euclidean distance between p1 and p2
def calculate_distance(point1, point2):
    return np.linalg.norm(np.array(point1) - np.array(point2))

def identify_dies(contours, min_die_area=55, max_die_area=450):
    
    #Identify dies with thresholding/contouring based on a minimum and maximum die area:
    die_offsets = []  # to store the center coordinates (offsets) of the bounding rectangles of the valid contours.
    for c in contours:
        area = cv2.contourArea(c)
        if min_die_area <= area <= max_die_area:
            x, y, w, h = cv2.boundingRect(c)
            center_x = x + w / 2
            center_y = y + h / 2
            die_offsets.append([center_x, center_y])
    return die_offsets # list of [center_x, center_y]

def get_skew_angle_min_area_rect(points):
    # use minimum bounding box
    rect = cv2.minAreaRect(points)
    # gets angle to correct for rotation
    angle = rect[-1]
    width, height = rect[1][0], rect[1][1]

    if width < height:
        angle = angle + 90
    return angle

def rotate_image(image, angle):
    # rotate image given the angle
    # positive angle -> rotate counterclockwise
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)

    # get rotation matrix
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    abs_cos = abs(M[0, 0])
    abs_sin = abs(M[0, 1])

    # get bounding dimensions of the rotated image
    new_w = int((h * abs_sin) + (w * abs_cos))
    new_h = int((h * abs_cos) + (w * abs_sin))

    # adjust rotation matrix to account for translation
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]

    rotated = cv2.warpAffine(image, M, (new_w, new_h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return rotated

def save_csv(data, path):
    # data: list of dictionaries containing paired contact information
    
    # csv headers
    headers = ['row', 'column', 'contact1_x', 'contact1_y', 'contact2_x', 'contact2_y']

    with open(path, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)
        writer.writeheader()
        for entry in data:
            writer.writerow({
                'row': entry['row'],
                'column': entry['contact1']['column'],
                'column': entry['contact2']['column'],
                'contact1_x': entry['contact1']['x'],
                'contact1_y': entry['contact1']['y'],
                'contact2_x': entry['contact2']['x'],
                'contact2_y': entry['contact2']['y']
            })
    # print(f"data saved as '{path}'")



In [3]:
def insert_cell_with_shift(lib, target_cell_name, insert_cell_name, translation_x=0, translation_y=0, rotation=0, magnification=0):
    # Insert one cell into another
    if target_cell_name not in lib.cells:
        print(f"Target cell '{target_cell_name}' not found in the library. Creating {target_cell_name}.")
        target_cell = lib.new_cell(f"{target_cell_name}")
    else:
        target_cell = lib.cells[target_cell_name]
    if insert_cell_name not in lib.cells:
        raise ValueError(f"Insert cell '{insert_cell_name}' not found in the library.")

    insert_cell = lib.cells[insert_cell_name]
    
    # Define transformation parameters
    translation = (translation_x, translation_y)
    rotation = rotation  # in degrees
    magnification = magnification

    # Create a reference to Cell-A with the specified transformations
    insert_cell_reference = gdspy.CellReference(
        insert_cell,
        origin=translation,
        rotation=rotation,
        magnification=magnification
    )

    # Add the reference to Cell-B
    target_cell.add(insert_cell_reference)
    # print(f"Inserted cell '{insert_cell_name}' into '{target_cell_name}'.")

In [4]:
def fix_column_outliers_past5(col, x_tolerance):
   #make route based on past 5 x coords, accounts for missing contact in columns for vertical right flexpath
    col_fixed = col.copy()
    
    for i in range(5, len(col_fixed)):  # start from 5 to have enough history
        prev_5_x = [col_fixed[j][0] for j in range(i - 5, i)]
        avg_x = sum(prev_5_x) / 5
        curr_x, curr_y = col_fixed[i]

        if abs(curr_x - avg_x) > x_tolerance:
            print(f"Outlier at index {i}: {curr_x:.2f} → Replacing with avg {avg_x:.2f}")
            col_fixed[i] = (avg_x, curr_y)

    return col_fixed


In [5]:
def flexpath_exists_near_point(x, y, flexpaths, x_tolerance=5, y_tolerance=1e3, precision=1e-3):
    """
    Check if any FlexPath polygon intersects a vertical strip of width 2*x_tolerance centered at x.
    y_tolerance defines the vertical range to consider (optional).
    """
    # Define the bounding rectangle (a vertical strip)
    search_box = [
        (x - x_tolerance, y - y_tolerance),
        (x + x_tolerance, y - y_tolerance),
        (x + x_tolerance, y + y_tolerance),
        (x - x_tolerance, y + y_tolerance)
    ]

    for fp in flexpaths:
        polys = fp.get_polygons()
        for poly in polys:
            # Check if any point in the search_box is inside the path polygon
            if any(gdspy.inside([pt], [poly], precision=precision)[0] for pt in search_box):
                return True
    return False

In [6]:
def majority_pad_side(col_history, column_idx):
    """
    Look at the last up to 3 assignments in col_history[column_idx]
    (each entry is 'left' or 'right'), and return the majority.
    Tie → 'left'
    """
    hist = col_history[column_idx][-3:]
    if not hist:
        return 'left'
    lefts  = hist.count('left')
    rights = hist.count('right')
    return 'left' if lefts >= rights else 'right'

In [7]:
def main():
    # input and output paths
    scan_dir = './input_images/csv_res'
    input_image_path = f'./input_images/TrueAdapt_uLED_Sample1_Scan0_half.png'
    threshold_image_path = f'{scan_dir}/threshold.png'
    rotated_path = f'{scan_dir}/rotated_threshold.png'

    #output images
    original_visualization_path = f'{scan_dir}/original_offsets_visualization.png'
    rows_visualization_path = f'{scan_dir}/rows_visualization.png'
    paired_visualization_path = f'{scan_dir}/paired_offsets_visualization.png'
    output_json_path = f'{scan_dir}/paired_contacts.json'
    output_csv_path = f'{scan_dir}/paired_contacts.csv'

    #gdspy file setup
    input_gds = 'uLED_input.gds'
    lib = gdspy.GdsLibrary(infile=input_gds)

    # Get the desired cell
    cell_name = 'Contact-Cell'
    for n in lib.cells:
        print(f"cell names: '{n}' ")
    if cell_name in lib.cells:
        selected_cell = lib.cells[cell_name]
        print(f"Cell '{cell_name}' selected.")
    else:
        selected_cell = lib.new_cell(cell_name)
        print(f"Cell '{cell_name}' not found. Creating new cell.")

    uled_cell = lib.cells['uLED-Array']
    
    


    
    img = cv2.imread(input_image_path)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # threshold
    _, bw = cv2.threshold(img_gray, 232, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)

    cv2.imwrite(threshold_image_path, bw)
    # print(f"thresholded image saved as '{threshold_image_path}'")

    contours, _ = cv2.findContours(bw, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)[-2:]
    die_offsets = identify_dies(contours, min_die_area=55, max_die_area=450)

    # convert die_offsets to numpy array for minAreaRect
    die_offsets_np = np.array(die_offsets).astype(np.float32)
    # print(f"number of detected contacts: {len(die_offsets_np)}")

    # --------- correct for rotation ---------
    angle = get_skew_angle_min_area_rect(die_offsets_np)
    # print(f"detected angle: {angle:.2f} degrees")
    # img_rotated = rotate_image(img, angle)
    img_rotated = img
    img_rotated_gray = cv2.cvtColor(img_rotated, cv2.COLOR_BGR2GRAY)

    # threshold rotated grayscale image
    _, bw_rotated = cv2.threshold(img_rotated_gray, 232, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    # rotation corrected image
    cv2.imwrite(rotated_path, bw_rotated)
    print(f"rotated image saved as '{rotated_path}'")

    #TODO: the gdspy file is flipped as compared to the actual image

    # defining region of interest for dies
    #TODO: for some reason, y2 corresponds to the top of the image and y1 refers to the bottom? So, the y coord decreases as we travel upwards in the img???
    x1, y1 = 2300, 14390 #top left, changing this y to lower val shaves the bottom of the imag but the top of the gdspy?
    x2, y2 = 14900, 4535 #bottom right, increasing y value shaves off the top of the image,
    y_min, y_max = sorted([y1, y2])
    x_min, x_max = sorted([x1, x2])


    # --- crop to ROI ---
    roi = bw_rotated[y_min:y_max, x_min:x_max]

    cv2.imwrite("cropped_roi.png", roi)

    # --- find contours inside that crop ---
    contours_roi, _ = cv2.findContours(
        roi,
        cv2.RETR_LIST,
        cv2.CHAIN_APPROX_SIMPLE
    )[-2:]

    # --- run die‐detection on the ROI contours ---
    die_offsets_roi = identify_dies(
        contours_roi,
        min_die_area=55,
        max_die_area=450
    )

    # --- shift them back into the ORIGINAL coordinate frame ---
    die_offsets_rotated = [
        (x + x_min, y + y_min)
        for (x, y) in die_offsets_roi
    ]



    # # find contours in rotated image
    # contours_rotated, _ = cv2.findContours(bw_rotated, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)[-2:]

    # # identify dies in rotated image
    # die_offsets_rotated = identify_dies(contours_rotated, min_die_area=55, max_die_area=450)
    # die_offsets_rotated_np = np.array(die_offsets_rotated).astype(np.float32)
    # print(f"number of detected contacts after rotation: {len(die_offsets_rotated_np)}")
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.6
    # --- visualize original offsets ---
    bw_color_original = cv2.cvtColor(bw_rotated.copy(), cv2.COLOR_GRAY2BGR)  
    marker_color = (0, 0, 255)  # red
    marker_radius = 5
    marker_thickness = -1  

    # draw red circles at each die offset and make gdspy vias using center
    for offset in die_offsets_rotated:
        center_x, center_y = offset
        center_coordinates = (int(center_x), int(center_y))
        cv2.circle(bw_color_original, center_coordinates, marker_radius, marker_color, marker_thickness)
        #placing contact shapes using center in gdspy
        contact_ref = gdspy.CellReference(selected_cell, origin=(center_x, center_y))
        uled_cell.add(contact_ref)
    
    cv2.imwrite(original_visualization_path, bw_color_original)
    print("image created b")




    # ---------- group contacts (indiviudal white spots, a pair of contacts makes an LED) into rows --------
    y_tolerance = 30  # pixels
    
    # sort die_offsets_rotated by y-coordinate
    die_offsets_rotated_sorted = sorted(die_offsets_rotated, key=lambda o: o[1])

    rows = []
    current_row = []
    current_y = None
    
    for offset in die_offsets_rotated_sorted:
        center_x, center_y = offset
        if not current_row:
            # start first row
            current_row.append(offset)
            current_y = center_y
            current_x = center_x
        else:
            # check if the current offset is within the y_tolerance of the current row
            if abs(center_y - current_y) <= y_tolerance:
                current_row.append(offset)
            else:
                # save the completed row and start a new one
                rows.append(current_row)
                current_row = [offset]
                current_y = center_y
    # append the last row if it's not empty
    if current_row:
        rows.append(current_row)

    # # number of rows detected and their number of contacts
    # print(f"number of rows detected: {len(rows)}")
    # for row_idx, row in enumerate(rows, start=0):
    #     print(f"row {row_idx}: {len(row)} contacts")
    
    
    cv2.imwrite(original_visualization_path, bw_color_original)
    # print(f"original offsets visualization saved as '{original_visualization_path}'")
            
   

    # ----- visualization: draw horizontal lines for each row ---------
    bw_color_rows = cv2.cvtColor(bw_rotated.copy(), cv2.COLOR_GRAY2BGR)  
    row_colors = [
        (0, 255, 255),  # yellow
        (0, 255, 0),    # green
        (255, 0, 0),    # blue
        (0, 255, 255),  # yellow
        (255, 0, 255),  # magenta
        (255, 255, 0),  # cyan
        (128, 0, 128),  # purple
        (0, 128, 128),  # teal
        (128, 128, 0),  # olive
        (0, 0, 128),    # maroon
    ]

    # draw red circles at each die offset on bw_color_rows
    for offset in die_offsets_rotated:
        center_x, center_y = offset
        center_coordinates = (int(center_x), int(center_y))
        cv2.circle(bw_color_rows, center_coordinates, marker_radius, marker_color, marker_thickness)

    # iterate through each row and draw a horizontal line at the median y-coordinate
    for row_idx, row in enumerate(rows, start=0):
        # use a new color for each row
        color = row_colors[(row_idx) % len(row_colors)]  

        # median y-coordinate of current row
        y_coords = [o[1] for o in row]
        median_y = int(np.median(y_coords))

        # draw horizontal line across at median_y
        cv2.line(bw_color_rows, (0, median_y), (bw_color_rows.shape[1], median_y), color, 2)

        # label row number 
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.6
        font_thickness = 2
        label = f'Row {row_idx}'
        cv2.putText(bw_color_rows, label, (10, median_y - 10), font, font_scale, color, font_thickness, cv2.LINE_AA)

        
        
            
        # color-code contacts based on row
        for offset in row:
            center_x, center_y = offset
            center_coordinates = (int(center_x), int(center_y))
            cv2.circle(bw_color_rows, center_coordinates, marker_radius, color, marker_thickness)
            # print("num cols: (lowest is 677 due to defects)", len(row))
    
       
    #removing contacts in each row that is a defect (x distance from previous contact too large)
    for ridx, row in enumerate(rows):
        row.sort(key=lambda o: o[0])
        if len(row) > 600:
            filtered = []
            for i in range(len(row)-1):
                center_x, center_y = row[i]
                nextx, nexty = row[i+1]
                if abs(nextx-center_x) <= 50:
                    filtered.append((center_x,center_y))
            # overwrite the old row with only the kept contacts
            rows[ridx] = filtered

    #sort rows
    for ridx, row in enumerate(rows):
        row.sort(key=lambda o: o[0])

    # save rows visualization image with color-coded contacts
    cv2.imwrite(rows_visualization_path, bw_color_rows)
    print(f"rows visualization with color-coded contacts saved as '{rows_visualization_path}'")



    # ------ pairing contacts within each row ------
    all_paired_offsets = []
    all_defected_offsets = []

    # distance threshold 
    distance_threshold = 25  # pixels

    # output: array containing paired contacts
    paired_contacts_output = []
    



    # #NEW METHOD: go through each row and construct artificial contacts if the spacing between any two contacts is too large
    print("running pairing algorithm...")
    paired_contacts_output = []
    firstx, firsty = rows[0][0]

    for row_idx, row in enumerate(rows):
        row = sorted(row, key=lambda p: p[0])
        i = 0

        # fill in any missing contacts by inserting fakes
        while i < len(row) - 1:
            x1, y1 = row[i]
            x2, y2 = row[i+1]

            # #if first contact in row x-coord is not the same x-coord as first row's first contact, then that means we are on the second contact, meaning its the right contact of the first pair
            # if i == 0 and firstx != x1:
            #     #insert fake left contact
            #     fake = (x1 - 15, y1)
            #     row.insert(0, fake)


            # decide which gap to expect 
            if i % 2 == 0:
                target_gap, tol = 15, (15, 19)
            else:
                target_gap, tol = 21, (35, 40)

            gap = abs(x2 - x1)
            print("gap: ", gap)
            if  gap <= tol[1]:
                i += 1
            else:
                # gap too large = insert a fake at exactly the lower‐bound, acceptable if gpa too small
                if gap < tol[0]:
                    print("gap too small")
                else:
                    print("gap too large")
                print("invalid spacing detected, adding fake")
                fake = (x1 + target_gap, y1)
                row.insert(i+1, fake)
                # x1,y1 will be the fake, and you can compare it to the next real
                i += 1
        
        #account for last contact in row that might be a pair
        xlast, ylast = row[i]
        fake = (xlast + 15, ylast)
        row.insert(i+1, fake)

        # creates pairs using repaired row
        col = 0
        while col < len(row) - 1:
            print("creating pairs..")
            x1, y1 = row[col]
            x2, y2 = row[col+1]
            paired_contacts_output.append({
                "row": row_idx,
                "contact1": {"x": x1, "y": y1, "column": col},
                "contact2": {"x": x2, "y": y2, "column": col+1}
            })
            col += 2
    print("finish pairing")

    
    

    # ------ GDSPY wire path routing ------
    #vertical gdspy routing (left contact) BASED ON COL INDEX USING PAIRED_CONTACTS_OUTPUT


    # 1. Group all left‐contacts by column index
    columns = defaultdict(list)
    for entry in paired_contacts_output:
        col = entry["contact1"]["column"]
        x, y = entry["contact1"]["x"], entry["contact1"]["y"]  # contact1 is the left‐most
        columns[col].append((x, y))

    # 2. For each column, sort, clean outliers, then draw the path
    existing_flexpaths = []  # if not already defined
    for col_idx in sorted(columns):
        pts = columns[col_idx]
        # sort top‐to‐bottom (or bottom‐to‐top, whichever makes sense for your layout)
        pts_sorted = sorted(pts, key=lambda p: p[1], reverse=True)
        # remove outliers based on the last 5 points in the column
        col_clean = fix_column_outliers_past5(pts_sorted, x_tolerance=5)
        
        if col_clean:
            flex = gdspy.FlexPath(
                col_clean,
                width=6,
                gdsii_path=True,
                layer=10
            )
            existing_flexpaths.append(flex)
            uled_cell.add(flex)

    
    m2_via_cell = '4um-M1-Via12-M2'
    via_cell = lib.cells[m2_via_cell]
    refs = []
    #vertical gdspy routing (right contact)
    # Add a vertical FlexPath for every contact2 in the paired contacts
    for pair in paired_contacts_output:
        # Extract the x and y coordinates of contact2 from the dictionary
        center_x = pair["contact2"]["x"]
        center_y = pair["contact2"]["y"]

        #add vias as well
        # create two references (no full copy of geometry!)
        refs.append( gdspy.CellReference(via_cell, (center_x, center_y+9)) )
        refs.append( gdspy.CellReference(via_cell, (center_x, center_y-9)) )
        # Create a vertical FlexPath at contact2's position
        rightpath = gdspy.FlexPath(
            [(center_x, center_y + 9), (center_x, center_y), (center_x, center_y - 9)],
            6, gdsii_path=True, layer=100
        )
        # Add the FlexPath to the top cell
        uled_cell.add(rightpath)
    for ref in refs:
        uled_cell.add(ref)


    #grab the left contact of every contact in a row
    for r in range(len(rows)):
        urowpath = []
        lrowpath = []
        row_pairs = [ entry for entry in paired_contacts_output if entry["row"] == r ]
        #add to path every contact2
        for pair in row_pairs:
            center_x = pair["contact2"]["x"]
            center_y = pair["contact2"]["y"]
            urowpath.append((center_x, center_y+9))
            lrowpath.append((center_x, center_y-9))
        uleftpath = gdspy.FlexPath(urowpath, 4, gdsii_path=True, layer=20)
        lleftpath = gdspy.FlexPath(lrowpath, 4, gdsii_path=True, layer=20)
        uled_cell.add(uleftpath)
        uled_cell.add(lleftpath)
            




    print(f"total Pairs: {len(all_paired_offsets)}")
    print(f"total defected contacts: {len(all_defected_offsets)}")

    with open(output_json_path, 'w') as f:
        json.dump(paired_contacts_output, f, indent=4)
    # print(f"paired contacts data saved as '{output_json_path}'")
    save_csv(paired_contacts_output, output_csv_path)
    
    # --------- visualize final output (pairs and defects) ------------
    bw_color_paired = cv2.cvtColor(bw_rotated.copy(), cv2.COLOR_GRAY2BGR) 
    line_color_paired = (0, 255, 0)       # green - paired lines
    circle_color_defected = (255, 0, 0)   # blue - defected contacts
    font_color = (0, 0, 255)              # red - r, c labels
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.3                          
    font_thickness = 1                        
    marker_radius = 5
    marker_thickness = -1  # filled circle

    
    # draw lines for each pair and annotate with their distance
    midpoints = []


    # First pass: draw each pair and record its midpoint
    for pair in paired_contacts_output:
        c1 = pair["contact1"]; c2 = pair["contact2"]
        p1 = (int(c1["x"]), int(c1["y"]))
        p2 = (int(c2["x"]), int(c2["y"]))

        # draw the pair‐line as before
        cv2.line(bw_color_paired, p1, p2, line_color_paired, 3)



        # compute midpoint
        mid = ((p1[0] + p2[0]) // 2, (p1[1] + p2[1]) // 2)
        midpoints.append(mid)

        # annotate with pair‐distance if you like
        dist = math.hypot(p2[0] - p1[0], p2[1] - p1[1])
        cv2.putText(
            bw_color_paired,
            f"{dist:.1f}",
            (mid[0] - 10, mid[1] - 10),
            font, font_scale, font_color, font_thickness, cv2.LINE_AA
    )

        rect = lib.cells['uLED-Outline']
        ref  = gdspy.CellReference(
            rect,
            origin=mid,
            rotation=0,
            magnification=1.0
        )
        uled_cell.add(ref)

    for i in range(len(midpoints) - 1):
        if i%2 == 0:
            m1 = midpoints[i]
            m2 = midpoints[i+1]

            # draw line between midpoints
            cv2.line(bw_color_paired, m1, m2, (255,255,0), 2)

            # compute midpoint‐to‐midpoint distance
            dx = m2[0] - m1[0]
            dy = m2[1] - m1[1]
            d_mid = math.hypot(dx, dy)

            # find text position (center of the segment)
            mid_mid = ((m1[0] + m2[0]) // 2, (m1[1] + m2[1]) // 2)

            # draw the distance label
            cv2.putText(
                bw_color_paired,
                f"{d_mid:.1f}",
                (mid_mid[0] - 20, mid_mid[1] - 20),
                font, font_scale, (255,255,0), font_thickness, cv2.LINE_AA
            )

    # draw defected contacts with blue circles
    for defected_contact in all_defected_offsets:
        center_x, center_y = defected_contact
        center_coordinates = (int(center_x), int(center_y))
        cv2.circle(bw_color_paired, center_coordinates, marker_radius, circle_color_defected, marker_thickness)

    cv2.imwrite(paired_visualization_path, bw_color_paired)
    print(f"paired offsets visualization saved as '{paired_visualization_path}'")
    lib.write_gds('finalrouting.gds')

if __name__ == "__main__":
    main()

# this script has a bug where if one contact is missing, it throws off the pair matching as visualized byt he rectangles covering every 2 contacts. As visualized in the image, currently, when a contaact is missing, the pair is skipped and a pair is made with the next two available contacts, instead of making a pair with a nonexistent contact. I want the script to modify the pairs_contacts_output to use the pair above's x coord for the left contact if it detects the left contact is missing. Essentially, I want a rectangle to encapsulate a pair even if a contact for the pair is missing.

cell names: '$$$CONTEXT_INFO$$$' 
cell names: 'CIRCLE' 
cell names: 'CIRCLE$5' 
cell names: 'CIRCLE$3' 
cell names: 'CIRCLE$1' 
cell names: '4um-M1-Via12-M2' 
cell names: 'uLED-Array' 
cell names: 'Contact-Cell' 
cell names: 'uLED-Outline' 
Cell 'Contact-Cell' selected.
rotated image saved as './input_images/csv_res/rotated_threshold.png'
image created b
rows visualization with color-coded contacts saved as './input_images/csv_res/rows_visualization.png'
running pairing algorithm...
gap:  14.5
gap:  21.5
gap:  35.5
gap too large
invalid spacing detected, adding fake
gap:  20.5
gap:  15.5
gap:  20.5
gap:  15.5
gap:  21.0
gap:  36.5
gap too large
invalid spacing detected, adding fake
gap:  21.5
gap:  15.5
gap:  20.5
gap:  36.5
gap too large
invalid spacing detected, adding fake
gap:  21.5
gap:  15.5
gap:  20.5
gap:  15.5
gap:  21.0
gap:  15.0
gap:  21.5
gap:  36.0
gap too large
invalid spacing detected, adding fake
gap:  21.0
gap:  15.5
gap:  20.5
gap:  15.5
gap:  20.5
gap:  15.5
gap:  2

/var/folders/vs/q5nlgtbd0h928z1tq9v9s8440000gq/T/ipykernel_31123/3813584157.py:617: UserWarning: [GDSPY] Properties with size larger than 128 bytes are not officially supported by the GDSII specification.  This file might not be compatible with all readers.
  lib.write_gds('finalrouting.gds')
